# Kaggle SSH · Cloudflare + Pinggy

Remote SSH into a **running Kaggle notebook** using either **Cloudflare Quick Tunnel** or **Pinggy**. Includes a small control panel, process management, and diagnostics.

**Usage:** Enable **Internet** in Kaggle notebook settings. Run the setup cell, paste your **SSH public key** (never your private key), select a tunnel backend, and press **Start**. You can start **Both**, compare them, or stop everything.

> **Important:** Kaggle may restrict remote access or long-running services; follow its current terms. A tunnel does not prevent session termination. Quick/free tunnels are ephemeral and may change address on restart. This notebook intentionally disables SSH password logins; keep public hostnames private when possible.

**What this notebook does not do:** bypass Kaggle quotas, keep sessions alive, or expose the Jupyter HTTP server. SSH is bound to `127.0.0.1:2222` inside the notebook environment.

## 1 · Setup SSH and tunnel utilities

Uses Python `subprocess.Popen` for persistent processes, not a `timeout` wrapper. Installs **no pip dependencies** for the SSH/tunnel backend; the optional control panel uses `ipywidgets` when present. Cloudflared is downloaded only if needed. `xxd` is unnecessary: Python checks the executable's ELF magic bytes directly.

In [ ]:
from __future__ import annotations
import os, re, shlex, shutil, socket, subprocess, threading, time, urllib.request, platform
from pathlib import Path

WORK = Path('/tmp/kaggle-ssh')
WORK.mkdir(parents=True, exist_ok=True)
PORT = 2222
CLOUDFLARED = WORK / 'cloudflared'
PROCESSES = {}
LOGS = {k: WORK / (k + '.log') for k in ('sshd', 'cloudflare', 'pinggy')}


def run(args, *, check=True, **kwargs):
    return subprocess.run(args, check=check, text=True, capture_output=True, **kwargs)


def binary_info(path):
    p = Path(path)
    return {'path': str(p), 'bytes': p.stat().st_size,
            'magic_hex': p.open('rb').read(8).hex(),
            'is_elf': p.open('rb').read(4) == b'\x7fELF'}


def ensure_cloudflared():
    if CLOUDFLARED.exists() and binary_info(CLOUDFLARED)['is_elf']:
        return CLOUDFLARED
    arch = platform.machine().lower()
    suffix = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(arch)
    if not suffix:
        raise RuntimeError(f'Unsupported CPU architecture: {arch}')
    url = 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-' + suffix
    temp = CLOUDFLARED.with_suffix('.download')
    try:
        urllib.request.urlretrieve(url, temp)
        if not binary_info(temp)['is_elf']:
            raise RuntimeError('Downloaded cloudflared is not an ELF executable')
        temp.chmod(0o755)
        temp.replace(CLOUDFLARED)
    finally:
        temp.unlink(missing_ok=True)
    return CLOUDFLARED


def read_log(name, tail=3500):
    p = LOGS[name]
    return p.read_text(errors='replace')[-tail:] if p.exists() else '(no log yet)'


def alive(name):
    p = PROCESSES.get(name)
    return p is not None and p.poll() is None


def launch(name, args):
    if alive(name):
        return PROCESSES[name]
    with LOGS[name].open('w') as output:
        proc = subprocess.Popen(args, stdin=subprocess.DEVNULL, stdout=output,
                                stderr=subprocess.STDOUT, start_new_session=True)
    PROCESSES[name] = proc
    return proc


def stop(name):
    proc = PROCESSES.pop(name, None)
    if proc is None or proc.poll() is not None:
        return
    import signal
    try:
        os.killpg(proc.pid, signal.SIGTERM)
        proc.wait(timeout=5)
    except (ProcessLookupError, subprocess.TimeoutExpired):
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGKILL)
            proc.wait(timeout=5)


def await_port(port, timeout=8):
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        with socket.socket() as s:
            if s.connect_ex(('127.0.0.1', port)) == 0:
                return True
        time.sleep(0.2)
    return False

print('Workspace:', WORK)
print('sshd:', shutil.which('sshd') or '/usr/sbin/sshd (if installed)')


## 2 · SSH server (public-key authentication only)

Generate a key on your **own computer** if needed: `ssh-keygen -t ed25519`. Paste the contents of `~/.ssh/id_ed25519.pub` into the control panel. The Kaggle notebook **never needs your private key**.

In [ ]:
def start_sshd(public_key: str):
    pub = public_key.strip()
    if not re.fullmatch(r'(ssh-ed25519|ssh-rsa|ecdsa-sha2-nistp(?:256|384|521))\s+[A-Za-z0-9+/]+={0,3}(?:\s+[^\r\n]*)?', pub):
        raise ValueError('Paste a complete OpenSSH public key (e.g. ssh-ed25519 AAAA...). Never paste a private key.')
    sshd = shutil.which('sshd') or '/usr/sbin/sshd'
    if not Path(sshd).exists():
        raise RuntimeError('OpenSSH server missing. Install openssh-server in this environment first.')
    ssh_dir = WORK / 'ssh'
    ssh_dir.mkdir(mode=0o700, exist_ok=True)
    ssh_dir.chmod(0o700)
    authorized = ssh_dir / 'authorized_keys'
    authorized.write_text(pub + '\n')
    authorized.chmod(0o600)
    hostkey = ssh_dir / 'ssh_host_ed25519_key'
    if not hostkey.exists():
        run(['ssh-keygen', '-q', '-t', 'ed25519', '-N', '', '-f', str(hostkey)])
    config = WORK / 'sshd_config'
    config.write_text(f"""Port {PORT}
ListenAddress 127.0.0.1
HostKey {hostkey}
PidFile {WORK / 'sshd.pid'}
AuthorizedKeysFile {authorized}
PermitRootLogin prohibit-password
PasswordAuthentication no
KbdInteractiveAuthentication no
PubkeyAuthentication yes
UsePAM no
AllowTcpForwarding yes
X11Forwarding no
PermitTunnel no
PermitUserEnvironment no
PrintMotd no
Subsystem sftp internal-sftp
""")
    check = run([sshd, '-t', '-f', str(config)], check=False)
    if check.returncode:
        raise RuntimeError('Invalid sshd config: ' + check.stderr)
    if alive('sshd'):
        stop('sshd')  # ensures a changed public key is honored
    launch('sshd', [sshd, '-D', '-e', '-f', str(config)])
    if not await_port(PORT):
        raise RuntimeError('sshd did not start:\n' + read_log('sshd'))
    return f'SSH listening on 127.0.0.1:{PORT}; password login disabled.'


def start_cloudflare():
    if not alive('sshd'):
        raise RuntimeError('Start SSH first.')
    binary = ensure_cloudflared()
    launch('cloudflare', [str(binary), 'tunnel', '--url', f'ssh://127.0.0.1:{PORT}', '--no-autoupdate'])
    for _ in range(40):
        log = read_log('cloudflare', 12000)
        match = re.search(r'https://([a-z0-9-]+\.trycloudflare\.com)', log)
        if match:
            host = match.group(1)
            return ('Cloudflare: ' + host + '\n'
                    + 'Client command (cloudflared must be installed locally):\n'
                    + f"ssh -o 'ProxyCommand=cloudflared access ssh --hostname %h' root@{host}")
        if not alive('cloudflare'):
            raise RuntimeError('cloudflared exited:\n' + log[-1500:])
        time.sleep(0.5)
    raise RuntimeError('Cloudflare hostname not found yet. Inspect Cloudflare log.')


def start_pinggy():
    if not alive('sshd'):
        raise RuntimeError('Start SSH first.')
    if not shutil.which('ssh'):
        raise RuntimeError('SSH client missing in Kaggle runtime.')
    launch('pinggy', ['ssh', '-T', '-p', '443',
        '-o', 'StrictHostKeyChecking=accept-new',
        '-o', f'UserKnownHostsFile={WORK / "known_hosts"}',
        '-o', 'ServerAliveInterval=30', '-o', 'ServerAliveCountMax=3',
        '-o', 'ExitOnForwardFailure=yes',
        '-R', f'0:127.0.0.1:{PORT}', 'tcp@free.pinggy.io'])
    time.sleep(4)
    log = read_log('pinggy', 6000)
    if not alive('pinggy'):
        raise RuntimeError('Pinggy tunnel exited:\n' + log)
    return 'Pinggy started. Find its public TCP host and port in the log below:\n' + log[-2500:]


def start_all(public_key, backend):
    messages = [start_sshd(public_key)]
    for name, starter in [('cloudflare', start_cloudflare), ('pinggy', start_pinggy)]:
        if backend in (name, 'both'):
            try:
                messages.append(starter())
            except Exception as exc:
                messages.append(f'{name}: ERROR: {exc}')
    return '\n\n'.join(messages)


def stop_all():
    for name in ('cloudflare', 'pinggy', 'sshd'):
        stop(name)
    return 'Stopped processes launched by this notebook.'


## 3 · Control panel

The notebook UI uses `ipywidgets` when available. Otherwise use the plain Python commands in the next cell. **Both** runs two separate public tunnels to the same local SSH server.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display
    key_input = widgets.Textarea(description='Public key', placeholder='ssh-ed25519 AAAA... (public key only)',
                                 layout=widgets.Layout(width='100%', height='86px'), style={'description_width': '80px'})
    backend_input = widgets.ToggleButtons(options=[('Cloudflare', 'cloudflare'), ('Pinggy', 'pinggy'), ('Both', 'both')],
                                          description='Tunnel', style={'description_width': '80px'})
    start_button = widgets.Button(description='Start', button_style='success', icon='play')
    stop_button = widgets.Button(description='Stop all', button_style='warning', icon='stop')
    refresh_button = widgets.Button(description='Logs / status', icon='refresh')
    output = widgets.Output(layout={'border': '1px solid #888', 'max_height': '440px', 'overflow': 'auto'})

    def render_status(_=None):
        with output:
            output.clear_output(wait=True)
            print('STATUS:', '  '.join(f'{name}={"running" if alive(name) else "off"}' for name in LOGS))
            for name in ('cloudflare', 'pinggy', 'sshd'):
                print(f'\n--- {name} ---\n{read_log(name, 1800)}')

    def on_start(_):
        start_button.disabled = True
        try:
            with output:
                output.clear_output(wait=True)
                print(start_all(key_input.value, backend_input.value))
        except Exception as exc:
            with output:
                print(f'ERROR: {exc}')
        finally:
            start_button.disabled = False

    def on_stop(_):
        with output:
            output.clear_output(wait=True)
            print(stop_all())

    start_button.on_click(on_start)
    stop_button.on_click(on_stop)
    refresh_button.on_click(render_status)
    display(widgets.VBox([
        widgets.HTML('<h3 style="margin:4px 0">Kaggle SSH controls</h3>'),
        key_input, backend_input,
        widgets.HBox([start_button, stop_button, refresh_button]), output]))
except ImportError:
    print('ipywidgets unavailable; use the Python fallback below.')


## 4 · Plain Python fallback / diagnostics

If notebook widgets do not render, set your public key here and call `start_all`. To inspect a tunnel, use `read_log('cloudflare')` or `read_log('pinggy')`. The connection addresses are **temporary**.

For Pinggy, connect to the public TCP hostname and port shown in its output: `ssh -p PORT root@HOST`. For Cloudflare, install `cloudflared` on your local computer and use the generated `ProxyCommand`.

In [ ]:
# Uncomment if you prefer not to use the UI:
# PUBLIC_KEY = 'ssh-ed25519 AAAA... your-computer'
# print(start_all(PUBLIC_KEY, 'both'))

# Useful checks (never prints the contents of your authorized_keys):
print({name: ('running' if alive(name) else 'stopped') for name in LOGS})
print('Cloudflare binary:', binary_info(CLOUDFLARED) if CLOUDFLARED.exists() else 'not downloaded')
# print(read_log('cloudflare'))
# print(read_log('pinggy'))


## 5 · Cleanup

Click **Stop all** in the panel, or run the next cell. Notebook shutdown will also terminate these processes. Restarting the kernel loses the Python process handles, so if you need to restart the tunnel, restart the Kaggle session or inspect old processes first.

**Limitations:** Cloudflare Quick Tunnels have no uptime SLA; Pinggy's free service may impose limits. Confirm that remote SSH/tunnel use is permitted by the current notebook-provider rules. Do not put credentials or private SSH keys in shared notebooks.

In [ ]:
# Uncomment to stop processes created during this kernel session:
# print(stop_all())